In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
sys.path.append(
    "./modules/python-utils:./modules/ai-utils"
)
print(f"Current Python version: {sys.version}")

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path
import librosa
import numpy as np

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import *
from sj_ai_utils.evaluator.sclite_utils import *

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test-other/LibriSpeech/test-other/"
DESTINATION = "/workspaces/dev/output/LibriSpeechASRcorpus/sclient/whisper/test-other/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="int8")

In [ ]:
src = Path(SOURCE)
dest = Path(DESTINATION)
dest.mkdir(parents=True, exist_ok=True)

In [ ]:
def transcriber(flac:Path) -> list[TRNFormat]:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)
    segments, _= model.transcribe(audio, beam_size=5)
    trn_list = segments_to_sclite_trn(flac.stem, segments)
    return trn_list

In [ ]:
make_all_ref_and_hyp(src, dest, transcriber)
concat_trn_file(
    list(sorted(p for p in dest.rglob("*.ref.trn"))),
    dest / "concat.ref.trn"
)
concat_trn_file(
    list(sorted(p for p in dest.rglob("*.hyp.trn"))),
    dest / "concat.hyp.trn"
)

In [ ]:
output = sclite_trn_run(
    dest / "concat.ref.trn",
    dest / "concat.hyp.trn",
)

In [ ]:
parse_sclite_summary(output)

# {'num_sentences': 2939,
#  'num_words': 52343,
#  'correct_percent': 90.7,
#  'substitution_percent': 8.5,
#  'deletion_percent': 0.9,
#  'insertion_percent': 2.2,
#  'wer_percent': 11.5,
#  'sentence_error_percent': 75.1}